# 8호선 역세권 주간 생활인구(Y) 추정 실험

**공식:**  $\hat{Y}_{i} = C_{0} \cdot C_{i}^{0.65} \cdot K_{i}^{0.15} \cdot A_{i}^{0.25}$

선행연구 기반 Cobb-Douglas calibration:
- **c = 0.65** (C, 이용량) — Lee et al. (2021), Kim et al. (2022), Chen et al. (2019)
- **k = 0.15** (K, 수송력) — Van Oort et al. (2015), Lee et al. (2019) + 다중공선성 조정
- **a = 0.25** (A, 접근성) — Yang et al. (2022), Kim et al. (2022), Lee et al. (2021)
- 합 = 1.05 ≈ 1 (준 규모수익불변)

**입력 (Downloads 폴더):**
- `8호선_역정보_표.csv`
- `수도권전철8호선_역번호순_일평균승하차량_2008_2024_재생성.csv`
- `운행횟수_표_수정본.csv`

**출력:**
- `8호선_Yhat_실험결과.csv`
- `8호선_Yhat_bar.png`, `8호선_Yhat_K_scatter.png`, `8호선_공급활동_gap.png`

## 1. 라이브러리 임포트 및 한글 폰트 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

for name in ['Malgun Gothic', 'NanumGothic', 'Nanum Gothic', 'AppleGothic']:
    if any(name.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = name
        break
plt.rcParams['axes.unicode_minus'] = False

DOWN = Path(r'c:/Users/inwoo/Downloads')
DOWN

## 2. 입력 데이터 로드

In [ ]:
info = pd.read_csv(DOWN / '8호선_역정보_표.csv', encoding='utf-8')
cdf  = pd.read_csv(DOWN / '수도권전철8호선_역번호순_일평균승하차량_2008_2024_재생성.csv', encoding='utf-8')
kdf  = pd.read_csv(DOWN / '운행횟수_표_수정본.csv', encoding='utf-8')

# BOM 제거
info.columns = [c.lstrip('\ufeff') for c in info.columns]
cdf.columns  = [c.lstrip('\ufeff') for c in cdf.columns]
kdf.columns  = [c.lstrip('\ufeff') for c in kdf.columns]

print('역정보', info.shape, '| 승하차', cdf.shape, '| 운행현황', kdf.shape)
info.head(3)

## 3. 변수 구성 — C(i), K(i), A(i)

- **C(i)** = i역의 2024 일평균 승하차
- **K(i)** = i역의 수송력 지수 (기본 6량 × 322회 = 1.0, 환승 시 연결 노선 편성·운행 합산 가중)
- **A(i)** = i역의 (1 + 환승노선 수) × (1 / √D_CBD) 정규화 → [1, 3] 범위
- **D_CBD** = 잠실역(814) 기준 누적 거리 (km, 하한 0.5)

In [ ]:
# C
c24 = cdf[['역번호', '역명', '2024']].rename(columns={'2024': 'C'})

# K
K_BASE = 1.0
k_map = dict(zip(kdf['역번호'], kdf['K']))

df = info[['노선', '역번호', '역명', '접속노선', '역간거리', '누적거리', 'L', '소재지']].copy()
df = df.merge(c24[['역번호', 'C']], on='역번호', how='left')
df['K'] = df['역번호'].map(k_map).fillna(K_BASE)

# 환승 노선 수
df['T_flag']  = df['접속노선'].fillna('').str.strip().astype(bool).astype(int)
df['T_count'] = df['접속노선'].fillna('').apply(lambda s: len([x for x in s.split(',') if x.strip()]))

# D_CBD
CBD_STATION_NO = 814
cbd_cum = df.loc[df['역번호'] == CBD_STATION_NO, '누적거리'].iloc[0]
df['D_CBD'] = (df['누적거리'] - cbd_cum).abs().clip(lower=0.5)

# A
df['A_raw'] = (1 + df['T_count']) * (1.0 / np.sqrt(df['D_CBD']))
a_min, a_max = df['A_raw'].min(), df['A_raw'].max()
df['A'] = 1 + 2 * (df['A_raw'] - a_min) / (a_max - a_min)

df[['역번호', '역명', 'C', 'K', 'T_count', 'D_CBD', 'A']].head(10)

## 4. Cobb-Douglas 합성 공식 적용

In [ ]:
GAMMA   = 0.65   # c : 이용량 탄력성
BETA    = 0.15   # k : 수송력 탄력성
THETA   = 0.25   # a : 접근성 탄력성
C_CONST = 1.0    # 스케일 상수

df['Y_hat_raw'] = C_CONST * (df['C'] ** GAMMA) * (df['K'] ** BETA) * (df['A'] ** THETA)

# 가독성을 위해 C 중앙값의 2배 수준으로 스케일링
scale = (df['C'].median() * 2.0) / df['Y_hat_raw'].median()
df['Y_hat'] = df['Y_hat_raw'] * scale

df[['역명', 'C', 'K', 'A', 'Y_hat']].head(10)

## 5. 순위·잔차 지표 계산 및 결과표 저장

In [ ]:
df['Util']   = df['Y_hat'] / (df['C'] * df['K'])
df['C_rank'] = df['C'].rank(ascending=False, method='min').astype(int)
df['K_rank'] = df['K'].rank(ascending=False, method='min').astype(int)
df['Y_rank'] = df['Y_hat'].rank(ascending=False, method='min').astype(int)
df['Gap']    = df['Y_rank'] - df['K_rank']

out_tbl = df[['역번호', '역명', '접속노선', '소재지', 'C', 'K', 'T_count', 'D_CBD', 'A',
              'Y_hat', 'C_rank', 'K_rank', 'Y_rank']].copy()
out_tbl = out_tbl.sort_values('Y_rank').reset_index(drop=True)

out_csv = DOWN / '8호선_Yhat_실험결과.csv'
out_tbl.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('saved:', out_csv)

print('\n[TOP 5 by Y_hat]')
print(out_tbl.head(5)[['역명', 'C', 'K', 'A', 'Y_hat', 'Y_rank']].to_string(index=False))
print('\n[BOTTOM 5 by Y_hat]')
print(out_tbl.tail(5)[['역명', 'C', 'K', 'A', 'Y_hat', 'Y_rank']].to_string(index=False))

## 6. 시각화 ① — C 대 Ŷ 막대 그래프 (y축 로그)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8), dpi=140)
df_plot = df.sort_values('역번호').reset_index(drop=True)
x = np.arange(len(df_plot))
w = 0.4
ax.bar(x - w/2, df_plot['C'],     width=w, color='#868E96', label='C · 일평균 승하차 (2024)')
ax.bar(x + w/2, df_plot['Y_hat'], width=w, color='#1C7ED6', label=r'Y_hat · 역세권 주간 생활인구(추정)')
ax.set_xticks(x)
ax.set_xticklabels(df_plot['역명'], rotation=55, ha='right', fontsize=9)
ax.set_yscale('log')
ymax = max(df_plot[['C', 'Y_hat']].max()) * 1.5
ax.set_ylim(5000, ymax)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title(r'8호선 24개 역 — Y_hat = C$^{0.65}$ · K$^{0.15}$ · A$^{0.25}$ 적용 결과 (y축 로그)')
ax.set_ylabel('명 (log scale)')
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3, axis='y')
for i, v in enumerate(df_plot['Y_hat']):
    ax.text(x[i] + w/2, v * 1.04, f'{int(v):,}',
            ha='center', va='bottom', fontsize=7, color='#1C7ED6', rotation=90)
plt.tight_layout()
p1 = DOWN / '8호선_Yhat_bar.png'
plt.savefig(p1, dpi=140, bbox_inches='tight')
plt.show()
print('saved:', p1)

## 7. 시각화 ② — K vs Ŷ 산점도 (저활용 식별용)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9), dpi=140)
sizes = df['C'] / 500
ax.scatter(df['K'], df['Y_hat'], s=sizes, c='#1C7ED6', alpha=0.65, edgecolor='#1F3A5F')
for _, r in df.iterrows():
    ax.annotate(r['역명'], (r['K'], r['Y_hat']),
                textcoords='offset points', xytext=(6, 4), fontsize=8, color='#222')

k_sorted = np.linspace(df['K'].min(), df['K'].max(), 100)
coef = np.polyfit(df['K'], df['Y_hat'], 1)
ax.plot(k_sorted, coef[0] * k_sorted + coef[1], '--', color='#C92A2A', lw=1.5,
        label='K ~ Y_hat 평균 추세')
ax.set_xlabel('K · 수송력 지수 (환승 가중)')
ax.set_ylabel('Y_hat · 역세권 주간 생활인구 추정')
ax.set_title('공급(K) 대비 활동(Y_hat) — 추세선 아래 = 저활용 역세권')
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
plt.tight_layout()
p2 = DOWN / '8호선_Yhat_K_scatter.png'
plt.savefig(p2, dpi=140, bbox_inches='tight')
plt.show()
print('saved:', p2)

## 8. 시각화 ③ — K→Ŷ 회귀 잔차 순위 (저활용 TOP 5)

In [ ]:
a_coef, b_coef = np.polyfit(df['K'], df['Y_hat'], 1)
df['Y_hat_trend'] = a_coef * df['K'] + b_coef
df['Residual']    = df['Y_hat'] - df['Y_hat_trend']

fig, ax = plt.subplots(figsize=(13, 8), dpi=140)
df_res = df.sort_values('Residual').reset_index(drop=True)
colors = ['#FA5252' if r < 0 else '#0CA678' for r in df_res['Residual']]
ax.barh(df_res['역명'], df_res['Residual'], color=colors)
ax.axvline(0, color='#222', lw=0.8)
ax.set_xlabel('Residual = Y_hat - (a·K + b)     >> 음수=공급 대비 저활용 · 양수=고활용')
ax.set_title('8호선 — 공급(K) 대비 활동(Y_hat) 잔차 순위')
ax.grid(True, axis='x', alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
for i, val in enumerate(df_res['Residual']):
    ax.text(val + (800 if val >= 0 else -800), i, f'{int(val):+,}',
            ha='left' if val >= 0 else 'right', va='center', fontsize=9)
plt.tight_layout()
p3 = DOWN / '8호선_공급활동_gap.png'
plt.savefig(p3, dpi=140, bbox_inches='tight')
plt.show()
print('saved:', p3)

print('\n[공급 대비 저활용 TOP 5 - Residual 최하]')
print(df_res.head(5)[['역명', 'C', 'K', 'Y_hat', 'Y_hat_trend', 'Residual']].to_string(index=False))